# Backprop lab: gradients by hand and by machine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/backprop.ipynb)

Compute gradients three ways (nudging, by hand, with a tiny autograd engine), then use them to train a small neural network. Companion to [Calculus & backprop](../prereqs/backprop.md). Pure Python, runs in seconds.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

import matplotlib.pyplot as plt
from s2a.autograd import Value, MLP, fit

## 1. A derivative is "what happens if I nudge this?"

Take `L = (a*b + c) * a`. Nudge each input by a tiny amount and see how much `L` moves.

In [ ]:
def L(a, b, c):
    return (a * b + c) * a

a, b, c, h = 2.0, -3.0, 10.0, 1e-6
print("L =", L(a, b, c))
print("dL/da ~", round((L(a + h, b, c) - L(a, b, c)) / h, 4))
print("dL/db ~", round((L(a, b + h, c) - L(a, b, c)) / h, 4))
print("dL/dc ~", round((L(a, b, c + h) - L(a, b, c)) / h, 4))

That took one extra run of `L` per input. A model with a billion weights would need a billion extra runs per step. Backpropagation gets every gradient in **one** backward pass.

## 2. The same gradients by backpropagation

By hand: `L = a²b + ac`, so `dL/da = 2ab + c = -2`, `dL/db = a² = 4`, `dL/dc = a = 2`.

In [ ]:
a, b, c = Value(2.0), Value(-3.0), Value(10.0)
out = (a * b + c) * a
out.backward()
print("L =", out.data)
print("dL/da =", a.grad, "| dL/db =", b.grad, "| dL/dc =", c.grad)

## 3. Look inside: the computation graph

Every `Value` remembers what it was made from. `backward()` walks this graph from the result to the inputs, multiplying local derivatives along the way (the chain rule).

In [ ]:
def show(v, depth=0):
    print("    " * depth + f"{v._op or 'input':>5}  value {v.data:>6.2f}  grad {v.grad:>6.2f}")
    for p in v._parents:
        show(p, depth + 1)

show(out)

`a` appears twice (value 2.00) because it's used twice. Its total gradient is the **sum** over both paths: that's why the engine uses `+=`.

## 4. Train a neural network with it

XOR: output 1 when exactly one input is 1. No straight line separates the two classes, so it needs a hidden layer. The network has 17 weights.

In [ ]:
xs = [[0, 0], [0, 1], [1, 0], [1, 1]]
ys = [0, 1, 1, 0]

model = MLP(2, [4, 1], seed=0)
print("before:", [round(model(x).data, 2) for x in xs])
losses = fit(model, xs, ys, steps=300, lr=0.2)
print("after: ", [round(model(x).data, 2) for x in xs], "target:", ys)

plt.semilogy(losses); plt.xlabel("step"); plt.ylabel("loss (log scale)"); plt.show()

## 5. The learning rate

Each step moves every weight by `learning rate × gradient`. Too small is slow; too large overshoots, and the loss can blow up instead of falling.

In [ ]:
for lr in [0.01, 0.05, 0.2, 0.5]:
    net, curve = MLP(2, [4, 1], seed=0), []
    while len(curve) < 300 and (not curve or curve[-1] < 100):   # stop early if it blows up
        curve += fit(net, xs, ys, steps=5, lr=lr)
    plt.semilogy(curve, label=f"lr {lr}")
    status = "diverged" if curve[-1] >= 100 else f"final loss {curve[-1]:.5f}"
    print(f"lr {lr:>4}: {status} after {len(curve)} steps")
plt.xlabel("step"); plt.ylabel("loss (log scale)"); plt.legend(); plt.show()

## Try it

- Add a `sigmoid` method to `Value` (subclass it, or edit a copy). What's its local derivative?
- Change the network to `MLP(2, [1])`: no hidden layer. Can it learn XOR? Why not?
- Try different seeds. Does every starting point reach the same solution?
- Count the `Value` objects created in one forward pass of the network. Why would this approach be far too slow for a real model, and what do PyTorch tensors do instead?